In [ ]:
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
!nvidia-smi

In [ ]:
# This cell mounts your Google Drive to the Colab environment, making your Drive files accessible in this notebook.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# This cell extracts the 'NSL.zip' dataset from your Google Drive to the Colab environment.
import zipfile
import os

# Define the path to the zip file in Google Drive
zip_file_path = '/content/drive/MyDrive/NSL.zip'

# Define the directory where the contents will be extracted
extraction_dir = '/content/'

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_dir, exist_ok=True)

try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extraction_dir)
    print(f

In [ ]:
# 1. Install dependencies
!pip install -q --upgrade ultralytics albumentations opencv-python-headless tensorboard

In [ ]:
# This script is designed to split an object detection dataset into training, validation, and test sets.
# It organizes images and their corresponding YOLO-formatted label files into the strict directory structure
# required for YOLO models (e.g., images/train, labels/train, etc.). The script ensures a reproducible
# stratified split based on defined ratios (e.g., 70% train, 15% validation, 15% test).
import os
import shutil
import random
from pathlib import Path

# ==========================================
# CONFIGURATION
# ==========================================
# Where your Label Studio export is (mixed images and .txt files)
SOURCE_DIR = r"/content/NSL"

# Where the perfectly split YOLO dataset will be built
OUTPUT_DIR = r"/content/split_dataset"

# Your split ratio (70% Train, 15% Validation, 15% Test)
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
# Test ratio is automatically the remaining 15%

# ==========================================
# EXECUTION
# ==========================================
def split_yolo_dataset(source, output, train_r, val_r):
    print("Starting Object Detection Stratified Split...")

    # 1. Create the strict YOLO folder structure
    subdirs = ['images/train', 'images/val', 'images/test',
               'labels/train', 'labels/val', 'labels/test']

    for subdir in subdirs:
        os.makedirs(os.path.join(output, subdir), exist_ok=True)

    # 2. Find all images and pair them with their .txt labels
    source_path = Path(source)
    image_dir = source_path / "images" if (source_path / "images").exists() else source_path
    label_dir = source_path / "labels" if (source_path / "labels").exists() else source_path
    valid_pairs = []
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    # Look for all common image types
    for img_path in image_dir.rglob('*'):
        if img_path.suffix.lower() not in image_extensions:
            continue

        txt_path = label_dir / f"{img_path.stem}.txt"

        # Only add to our list if BOTH the image and the bounding box file exist
        if txt_path.exists():
            valid_pairs.append((img_path, txt_path))

    total_files = len(valid_pairs)
    print(f"Found {total_files} perfectly matched Image + Label pairs.")

    if total_files == 0:
        print("Error: No paired data found. Check your source folder.")
        return

    # 3. Shuffle the dataset randomly (Crucial for preventing bias)
    random.seed(42) # Keeps the randomness consistent for your academic methodology
    random.shuffle(valid_pairs)

    # 4. Calculate the exact index to slice the list
    train_end = int(total_files * train_r)
    val_end = train_end + int(total_files * val_r)

    train_pairs = valid_pairs[:train_end]
    val_pairs = valid_pairs[train_end:val_end]
    test_pairs = valid_pairs[val_end:]

    print(f"Splitting: Train({len(train_pairs)}), Val({len(val_pairs)}), Test({len(test_pairs)})")

    # 5. Helper function to copy files to their new specific homes
    def copy_files(pairs, split_name):
        for img_path, txt_path in pairs:
            # Move Image
            shutil.move(img_path, os.path.join(output, f"images/{split_name}", img_path.name))
            # Move Label
            shutil.move(txt_path, os.path.join(output, f"labels/{split_name}", txt_path.name))

    # 6. Execute the moving
    copy_files(train_pairs, 'train')
    copy_files(val_pairs, 'val')
    copy_files(test_pairs, 'test')

    print(f"\nSuccess! Dataset strictly formatted for Object Detection at: {OUTPUT_DIR}")

# Run the function
split_yolo_dataset(SOURCE_DIR, OUTPUT_DIR, TRAIN_RATIO, VAL_RATIO)

In [ ]:
# This script performs robust data augmentation on a hand gesture dataset for YOLO object detection.
# It uses the Albumentations library to apply transformations (e.g., rotation, brightness, contrast, scale, blur)
# while carefully preserving the semantic meaning of hand gestures by avoiding operations like flipping.
# The script generates augmented image-label pairs and provides a summary of class counts.
from __future__ import annotations

import argparse
import sys
import time
from collections import defaultdict
from pathlib import Path

try:
    import cv2
    import albumentations as A
    from tqdm import tqdm
except ImportError as e:
    print(f"[ERROR] Missing required library: {e}")
    print("Please install dependencies: pip install opencv-python albumentations tqdm")
    sys.exit(1)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Robust data augmentation for hand gesture dataset using Albumentations with YOLO bounding box sync.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument(
        "--nsl-dir",
        "-d",
        type=Path,
        default=Path("/content/split_dataset"),
        help="Path to root Split directory containing 'images/' and 'labels/' subfolders.",
    )
    parser.add_argument(
        "--num-aug",
        "-n",
        type=int,
        default=3,
        help="Number of augmented variations to generate per original image.",
    )
    parser.add_argument(
        "--min-visibility",
        type=float,
        default=0.8,
        help="Minimum bounding box visibility fraction (0.8 means max 20%% clipped allowed).",
    )
    parser.add_argument(
        "--log-interval",
        type=int,
        default=1000,
        help="Print a monitoring progress milestone log every N images.",
    )
    parser.add_argument(
        "--images-dir",
        type=Path,
        default=None,
        help="Custom path to input images folder (defaults to <nsl-dir>/images).",
    )
    parser.add_argument(
        "--labels-dir",
        type=Path,
        default=None,
        help="Custom path to input labels folder (defaults to <nsl-dir>/labels).",
    )
    parser.add_argument(
        "--output-images-dir",
        type=Path,
        default=None,
        help="Custom path to output augmented images folder (defaults to <nsl-dir>/augmented_images).",
    )
    parser.add_argument(
        "--output-labels-dir",
        type=Path,
        default=None,
        help="Custom path to output augmented labels folder (defaults to <nsl-dir>/augmented_labels).",
    )
    parser.add_argument(
        "--yaml-path",
        type=Path,
        default=None,
        help="Path to dataset.yaml for mapping class names in summary table.",
    )
    return parser.parse_known_args()[0]


def format_time(seconds: float) -> str:
    """Format seconds into human-readable HHh MMm SSs format."""
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours > 0:
        return f"{hours}h {minutes:02d}m {secs:02d}s"
    return f"{minutes:02d}m {secs:02d}s"


def load_class_names(yaml_path: Path) -> dict[int, str]:
    """Parse dataset.yaml for human-readable class names."""
    names_map: dict[int, str] = {}
    if not yaml_path.exists():
        return names_map

    try:
        with open(yaml_path, "r", encoding="utf-8") as f:
            content = f.read()

        in_names = False
        for line in content.splitlines():
            stripped = line.strip()
            if stripped.startswith("names:"):
                in_names = True
                continue
            if in_names:
                if ":" in stripped and not stripped.startswith("#"):
                    parts = stripped.split(":", 1)
                    try:
                        cid = int(parts[0].strip())
                        cname = parts[1].strip().strip("'\"")
                        names_map[cid] = cname
                    except ValueError:
                        pass
                elif stripped and not stripped.startswith("#") and not stripped.startswith(" "):
                    in_names = False
    except Exception as e:
        print(f"[WARN] Failed to parse class names from '{yaml_path}': {e}")
    return names_map


def build_transform_pipeline(min_visibility: float = 0.8) -> A.Compose:
    """
    Build Albumentations pipeline strictly configured for hand gesture preservation:
    - Random Rotation: -15° to +15°
    - Brightness & Contrast: -20% to +20%
    - Scale/Zoom: 0.9x to 1.1x
    - Shift/Translation: Max 5%
    - Gaussian Blur / Noise: Kernel 3x3 / mild ISO noise
    - NO Flipping (Fliplr / Flipud disabled)
    - BboxParams: format='yolo', min_visibility=min_visibility (max 20% clipping allowed)
    """
    return A.Compose(
        [
            # Spatial transformations (Shift <= 5%, Scale 0.9x-1.1x, Rotation -15° to +15%)
            A.ShiftScaleRotate(
                shift_limit=0.05,
                scale_limit=0.1,
                rotate_limit=15,
                border_mode=cv2.BORDER_CONSTANT,
                value=(128, 128, 128),
                p=0.8,
            ),
            # Color & Contrast (Brightness/Contrast ±20%)
            A.RandomBrightnessContrast(
                brightness_limit=0.2,
                contrast_limit=0.2,
                p=0.7,
            ),
            # Slight Gaussian blur or noise
            A.OneOf(
                [
                    A.GaussianBlur(blur_limit=(3, 3), p=0.5),
                    A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
                ],
                p=0.5,
            ),
        ],
        bbox_params=A.BboxParams(
            format="yolo",
            label_fields=["class_labels"],
            min_visibility=min_visibility,
        ),
    )


def read_yolo_labels(lbl_path: Path) -> tuple[list[list[float]], list[int]]:
    """Read YOLO format bounding box file: <class_id> <xc> <yc> <w> <h>."""
    bboxes: list[list[float]] = []
    class_labels: list[int] = []

    if not lbl_path.exists():
        return bboxes, class_labels

    with open(lbl_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                try:
                    cls_id = int(float(parts[0]))
                    xc, yc, w, h = map(float, parts[1:])
                    # Clamp normalized coordinates to [0.0, 1.0] for Albumentations
                    xc = max(0.0001, min(0.9999, xc))
                    yc = max(0.0001, min(0.9999, yc))
                    w = max(0.0001, min(0.9999, w))
                    h = max(0.0001, min(0.9999, h))

                    bboxes.append([xc, yc, w, h])
                    class_labels.append(cls_id)
                except ValueError:
                    continue

    return bboxes, class_labels


def write_yolo_labels(lbl_path: Path, bboxes: list[list[float]], class_labels: list[int]) -> None:
    """Write updated YOLO format label text file."""
    lines = []
    for (xc, yc, w, h), cls_id in zip(bboxes, class_labels):
        xc = max(0.0, min(1.0, xc))
        yc = max(0.0, min(1.0, yc))
        w = max(0.0, min(1.0, w))
        h = max(0.0, min(1.0, h))
        lines.append(f"{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

    with open(lbl_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + ("\n" if lines else ""))


def process_augmentation(
    images_dir: Path,
    labels_dir: Path,
    out_images_dir: Path,
    out_labels_dir: Path,
    num_aug: int = 3,
    min_visibility: float = 0.8,
    log_interval: int = 1000,
    yaml_path: Path | None = None,
) -> None:
    if not images_dir.exists():
        print(f"[ERROR] Input images directory '{images_dir}' does not exist.")
        return

    out_images_dir.mkdir(parents=True, exist_ok=True)
    out_labels_dir.mkdir(parents=True, exist_ok=True)

    # Collect input image files
    image_files = sorted(
        f for f in images_dir.iterdir()
        if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
    )
    total_images = len(image_files)

    if not image_files:
        print(f"[WARN] No image files found in '{images_dir}'.")
        return

    # Load class name mapping if yaml provided
    class_names = load_class_names(yaml_path) if yaml_path else {}

    print("=" * 75)
    print(" Hand Gesture Dataset Augmenter (Albumentations) ")
    print("=" * 75)
    print(f" Input Images Directory  : {images_dir.resolve()}")
    print(f" Input Labels Directory  : {labels_dir.resolve()}")
    print(f" Output Images Directory : {out_images_dir.resolve()}")
    print(f" Output Labels Directory : {out_labels_dir.resolve()}")
    print(f" Total Real Images Found : {total_images:,}")
    print(f" Augmentations Per Image : {num_aug}x")
    print(f" Target Total Generated  : {total_images * num_aug:,} augmented images")
    print(f" Max Clipping Allowed    : {(1 - min_visibility) * 100:.0f}% (Min Visibility: {min_visibility})")
    print(" Monitoring Log Interval : Every {:,} images".format(log_interval))
    print(" CRITICAL RULE          : Horizontal & Vertical Flipping DISABLED")
    print("-" * 75)

    transform = build_transform_pipeline(min_visibility=min_visibility)

    orig_class_counts: dict[int, int] = defaultdict(int)
    aug_class_counts: dict[int, int] = defaultdict(int)
    skipped_empty_aug = 0
    total_aug_generated = 0

    start_time = time.time()
    pbar = tqdm(enumerate(image_files, start=1), total=total_images, desc="Augmenting Images", unit="img")

    for idx, img_file in pbar:
        lbl_file = labels_dir / f"{img_file.stem}.txt"
        bboxes, class_labels = read_yolo_labels(lbl_file)

        if not bboxes:
            continue

        # Count original class occurrences
        for cid in class_labels:
            orig_class_counts[cid] += 1

        img = cv2.imread(str(img_file))
        if img is None:
            continue

        # Generate N augmented variations
        for aug_idx in range(1, num_aug + 1):
            try:
                transformed = transform(image=img, bboxes=bboxes, class_labels=class_labels)
                aug_img = transformed["image"]
                aug_bboxes = transformed["bboxes"]
                aug_cls = transformed["class_labels"]

                # If all bboxes were clipped out by extreme transform, retry once or skip
                if not aug_bboxes:
                    skipped_empty_aug += 1
                    continue

                aug_stem = f"{img_file.stem}_aug{aug_idx}"
                out_img_path = out_images_dir / f"{aug_stem}{img_file.suffix.lower()}"
                out_lbl_path = out_labels_dir / f"{aug_stem}.txt"

                # Save augmented image and label file
                cv2.imwrite(str(out_img_path), aug_img)
                write_yolo_labels(out_lbl_path, aug_bboxes, aug_cls)

                for cid in aug_cls:
                    aug_class_counts[cid] += 1

                total_aug_generated += 1
            except Exception as e:
                print(f"[WARN] Error augmenting '{img_file.name}' (aug {aug_idx}): {e}")

        # Update tqdm live postfix stats
        aug_bbox_total = sum(aug_class_counts.values())
        pbar.set_postfix({
            "Generated": f"{total_aug_generated:,}",
            "AugBboxes": f"{aug_bbox_total:,}",
            "Clipped": skipped_empty_aug,
        })

        # Periodic Monitoring Progress Milestone Snapshot
        if log_interval > 0 and (idx % log_interval == 0 or idx == total_images):
            elapsed = time.time() - start_time
            rate = idx / elapsed if elapsed > 0 else 0
            eta = (total_images - idx) / rate if rate > 0 else 0
            pct = (idx / total_images) * 100
            tqdm.write(
                f"[MONITOR {pct:5.1f}%] Processed {idx:,}/{total_images:,} imgs | "
                f"Generated: {total_aug_generated:,} pairs | "
                f"Speed: {rate:.1f} img/s | Elapsed: {format_time(elapsed)} | ETA: {format_time(eta)}"
            )

    # Summary Table Display
    all_class_ids = sorted(set(orig_class_counts.keys()).union(aug_class_counts.keys()))

    print("\n" + "=" * 75)
    print(" AUGMENTATION DATASET SUMMARY TABLE ")
    print("=" * 75)
    print(f"{'Class ID':<10} | {'Class Label':<15} | {'Original Bboxes':<16} | {'Augmented Bboxes':<18} | {'Total Bboxes':<14}")
    print("-" * 75)

    total_orig_boxes = 0
    total_aug_boxes = 0

    for cid in all_class_ids:
        clabel = class_names.get(cid, f"Class {cid}")
        orig_c = orig_class_counts.get(cid, 0)
        aug_c = aug_class_counts.get(cid, 0)
        tot_c = orig_c + aug_c

        total_orig_boxes += orig_c
        total_aug_boxes += aug_c

        print(f"{cid:<10} | {clabel:<15} | {orig_c:<16} | {aug_c:<18} | {tot_c:<14}")

    print("-" * 75)
    print(f"{'TOTALS':<10} | {'33 Classes':<15} | {total_orig_boxes:<16} | {total_aug_boxes:<18} | {total_orig_boxes + total_aug_boxes:<14}")
    print("=" * 75)
    total_elapsed = time.time() - start_time
    print(f"\n[DONE] Successfully generated {total_aug_generated:,} augmented image-label pairs in {format_time(total_elapsed)} into '{out_images_dir}' and '{out_labels_dir}'.\n")


def main() -> None:
    args = parse_args()

    nsl_dir = args.nsl_dir
    images_dir = args.images_dir if args.images_dir else nsl_dir / "images"
    labels_dir = args.labels_dir if args.labels_dir else nsl_dir / "labels"
    out_images_dir = args.output_images_dir if args.output_images_dir else nsl_dir / "augmented_images"
    out_labels_dir = args.output_labels_dir if args.output_labels_dir else nsl_dir / "augmented_labels"
    yaml_path = args.yaml_path if args.yaml_path else nsl_dir / "dataset.yaml"
    if not yaml_path.exists():
        yaml_path = Path("dataset.yaml")

    process_augmentation(
        images_dir=images_dir,
        labels_dir=labels_dir,
        out_images_dir=out_images_dir,
        out_labels_dir=out_labels_dir,
        num_aug=args.num_aug,
        min_visibility=args.min_visibility,
        log_interval=args.log_interval,
        yaml_path=yaml_path,
    )


if __name__ == "__main__":
    main()

In [ ]:
# This script cleans up the environment by deleting the 'images' and 'labels' subdirectories
# within the '/content/NSL' directory. This is often done to remove previously extracted or
# processed data, ensuring a clean state before new data preparation steps like splitting or augmentation.
import shutil
import os

nsl_dir = '/content/NSL'
images_dir = os.path.join(nsl_dir, 'images')
labels_dir = os.path.join(nsl_dir, 'labels')

if os.path.exists(images_dir):
    shutil.rmtree(images_dir)
    print(f"Deleted: {images_dir}")
else:
    print(f"Directory not found: {images_dir}")

if os.path.exists(labels_dir):
    shutil.rmtree(labels_dir)
    print(f"Deleted: {labels_dir}")
else:
    print(f"Directory not found: {labels_dir}")

In [ ]:
# This script is designed for training a YOLO11n object detection model, performing thesis-grade evaluation,
# and integrating with TensorBoard for monitoring. It handles dynamic dataset path management, including
# moving data from Google Drive to local storage for faster access and updating the `dataset.yaml` file
# with correct paths for training, validation, and testing splits. After training, it evaluates the model
# on both validation and test sets and verifies the generation of various performance artifacts (plots, metrics, weights).
from __future__ import annotations

import argparse
import os
import sys
import time
from pathlib import Path
import shutil
import yaml # Added import for yaml
import subprocess # Added import for subprocess to run TensorBoard

_ultralytics_available = False
try:
    from ultralytics import YOLO
    _ultralytics_available = True
except ImportError:
    print("\n[CRITICAL ERROR] 'ultralytics' is not installed in the active environment. Please run the installation cell.\n")

# Default configuration paths (Updated to YOLO11n)
DEFAULT_DATA_YAML = Path("/content/drive/MyDrive/split_dataset") / "dataset.yaml"
DEFAULT_MODEL = "yolo11n.pt"


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Train YOLO11n model on split_dataset with TensorBoard logging and Master's/PhD thesis grade evaluation artifacts.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument(
        "--data",
        "-d",
        type=Path,
        default=DEFAULT_DATA_YAML,
        help="Path to YOLO dataset.yaml file.",
    )
    parser.add_argument(
        "--model",
        "-m",
        type=str,
        default=DEFAULT_MODEL,
        help="Starting model weights or model architecture (e.g., yolo11n.pt).",
    )
    parser.add_argument(
        "--epochs",
        "-e",
        type=int,
        default=50,
        help="Number of training epochs.",
    )
    parser.add_argument(
        "--imgsz",
        type=int,
        default=640,
        help="Training image resolution (e.g. 640 or 400).",
    )
    parser.add_argument(
        "--batch",
        "-b",
        type=int,
        default=-1, # Changed to -1 for AutoBatch
        help="Training batch size (-1 for AutoBatch).",
    )
    parser.add_argument(
        "--project",
        type=Path,
        default=Path("runs") / "detect",
        help="Ultralytics project output directory.",
    )
    parser.add_argument(
        "--name",
        type=str,
        default="train_yolo11n",
        help="Ultralytics run name for training outputs.",
    )
    parser.add_argument(
        "--device",
        default=None,
        help="Target device (e.g. '0' for CUDA GPU 0, 'cpu', or leave empty for auto-detect).",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=42,
        help="Random seed for reproducible thesis training.",
    )
    parser.add_argument(
        "--workers",
        type=int,
        default=16, # Increased workers for faster data loading
        help="Number of dataloader worker threads.",
    )
    parser.add_argument(
        "--cache",
        action="store_true",
        default=True,
        help="Cache images in RAM to eliminate disk I/O bottlenecks (2x-3x speedup).",
    )
    parser.add_argument(
        "--no-cache",
        action="store_false",
        dest="cache",
        help="Disable RAM caching if RAM space is limited.",
    )
    parser.add_argument(
        "--no-amp",
        action="store_true",
        help="Disable Automatic Mixed Precision (AMP).",
    )
    return parser.parse_known_args()[0]


def format_time(seconds: float) -> str:
    """Format seconds into HHh MMm SSs."""
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours > 0:
        return f"{hours}h {minutes:02d}m {secs:02d}s"
    return f"{minutes:02d}m {secs:02d}s"


def verify_thesis_artifacts(run_dir: Path) -> list[tuple[str, bool]]:
    """Check for existence of Master's / PhD thesis evaluation plots & metrics."""
    expected_artifacts = [
        ("weights/best.pt (Top Model Checkpoint)", run_dir / "weights" / "best.pt"),
        ("weights/last.pt (Final Epoch Checkpoint)", run_dir / "weights" / "last.pt"),
        ("F1_curve.png (F1 Score vs Confidence)", run_dir / "F1_curve.png"),
        ("PR_curve.png (Precision-Recall Curve)", run_dir / "PR_curve.png"),
        ("P_curve.png (Precision vs Confidence)", run_dir / "P_curve.png"),
        ("R_curve.png (Recall vs Confidence)", run_dir / "R_curve.png"),
        ("confusion_matrix.png (Confusion Matrix)", run_dir / "confusion_matrix.png"),
        ("confusion_matrix_normalized.png (Normalized Confusion Matrix)", run_dir / "confusion_matrix_normalized.png"),
        ("results.png (Loss & Metric Epoch Curves)", run_dir / "results.png"),
        ("results.csv (Numerical Loss/Metric Log)", run_dir / "results.csv"),
        ("train_batch0.jpg (Training Batch Overlay)", run_dir / "train_batch0.jpg"),
        ("val_batch0_labels.jpg (Val BBox Ground Truth)", run_dir / "val_batch0_labels.jpg"),
        ("val_batch0_pred.jpg (Val BBox Predictions)", run_dir / "val_batch0_pred.jpg"),
    ]

    artifact_status = []
    for name, path in expected_artifacts:
        artifact_status.append((name, path.exists()))

    return artifact_status


def run_yolo_training(args: argparse.Namespace) -> None:
    original_data_path = args.data.resolve()
    local_split_dataset_dir = Path("/content/split_dataset")
    local_data_yaml_path = local_split_dataset_dir / "dataset.yaml"

    # Check if the data is currently in Google Drive and needs to be moved locally
    if str(original_data_path.parent).startswith('/content/drive/MyDrive') and not local_split_dataset_dir.exists():
        print(f"[INFO] Moving dataset from Google Drive ({original_data_path.parent}) to local storage ({local_split_dataset_dir}) for faster access.")
        try:
            shutil.copytree(original_data_path.parent, local_split_dataset_dir)
            print(f"[SUCCESS] Dataset copied to local storage: {local_split_dataset_dir}")
            data_path = local_data_yaml_path # Update data_path to the local version
        except Exception as e:
            print(f"[ERROR] Failed to copy dataset to local storage: {e}. Proceeding with original path.")
            data_path = original_data_path
    elif local_split_dataset_dir.exists():
        print(f"[INFO] Using locally stored dataset at: {local_split_dataset_dir}")
        data_path = local_data_yaml_path
    else:
        data_path = original_data_path

    if not data_path.exists():
        found_yaml = False
        # Try current directory first
        if Path("dataset.yaml").exists():
            data_path = Path("dataset.yaml").resolve()
            found_yaml = True
        # Then try dataset_augmented directory
        elif (Path("dataset_augmented") / "dataset.yaml").exists():
            data_path = (Path("dataset_augmented") / "dataset.yaml").resolve()
            found_yaml = True
        # Check if dataset.yaml is in the original NSL directory (/content/NSL) and copy it
        elif (Path("/content/NSL") / "dataset.yaml").exists():
            source_yaml_path = Path("/content/NSL") / "dataset.yaml"
            # Ensure the target directory for data_path exists
            data_path.parent.mkdir(parents=True, exist_ok=True)
            try:
                shutil.copy(source_yaml_path, data_path)
                print(f"[INFO] Copied {source_yaml_path} to {data_path}")
                found_yaml = True
            except Exception as e:
                print(f"[WARN] Failed to copy dataset.yaml from /content/NSL: {e}")
                found_yaml = False

        if not found_yaml:
            raise FileNotFoundError(f"Could not find dataset.yaml configuration at: {args.data} or other known locations.")

    # --- Start of new logic to update dataset.yaml paths ---
    # If dataset.yaml exists (either initially or was copied), update its paths
    if data_path.exists():
        try:
            with open(data_path, 'r') as f:
                dataset_config = yaml.safe_load(f)

            # Update paths to be relative to the dataset.yaml's new location
            # This now points to the local split_dataset directory
            dataset_config['path'] = str(data_path.parent)
            dataset_config['train'] = 'images/train'
            dataset_config['val'] = 'images/val'
            dataset_config['test'] = 'images/test' if 'test' in dataset_config else dataset_config.get('val', 'images/val') # Ensure test path if it exists, default to val

            # Write the updated YAML back to the file
            with open(data_path, 'w') as f:
                yaml.safe_dump(dataset_config, f)
            print(f"[INFO] Updated dataset.yaml at {data_path} with correct split paths.")
        except Exception as e:
            print(f"[WARN] Failed to update dataset.yaml paths: {e}")
            # If updating fails, the original FileNotFoundError will likely occur later if paths are wrong
    # --- End of new logic ---

    run_dir = args.project.resolve() / args.name

    print("=" * 80)
    print(" YOLO11 Master's / PhD Thesis Model Trainer & TensorBoard Evaluator ")
    print("=" * 80)
    print(f" Model Architecture     : {args.model}")
    print(f" Dataset Configuration  : {data_path}")
    print(f" Training Epochs        : {args.epochs}")
    print(f" Image Resolution       : {args.imgsz}x{args.imgsz}")
    print(f" Batch Size             : {args.batch}")
    print(f" Run Directory          : {run_dir}")
    print(f" Random Seed            : {args.seed}")
    print("-" * 80)

    # Load starting YOLO model architecture/weights
    print(f"[INFO] Initializing YOLO model '{args.model}'...")
    model = YOLO(args.model)

    start_time = time.time()

    # Start TensorBoard in a separate process
    log_dir = args.project.resolve()
    print(f"\n[INFO] Starting TensorBoard logging for: {log_dir}")
    try:
        # Launch TensorBoard in the background
        tb_process = subprocess.Popen(
            ['tensorboard', '--logdir', str(log_dir), '--host', '0.0.0.0', '--port', '6006'],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE
        )
        print(f"[INFO] TensorBoard started (PID: {tb_process.pid}). Access it via the 'Ports' tab in Colab or at http://localhost:6006 (may take a moment to load).")
    except FileNotFoundError:
        print("[WARN] TensorBoard command not found. Please ensure it's installed or in your PATH.")
    except Exception as e:
        print(f"[WARN] Could not start TensorBoard: {e}")

    # Train YOLO11 model
    print("\n[INFO] Starting model training...")
    results = model.train(
        data=str(data_path),
        epochs=args.epochs,
        imgsz=args.imgsz,
        batch=args.batch,
        project=str(args.project.resolve()),
        name=args.name,
        exist_ok=True,
        plots=True,          # Generates PR_curve, F1_curve, confusion_matrix, results.png
        save=True,           # Saves best.pt and last.pt weights
        val=False,           # Changed to False for faster training per epoch
        seed=args.seed,
        workers=args.workers,
        device=args.device if args.device else None,
        cache=args.cache,    # Caches images in RAM to eliminate disk reading bottleneck
        amp=not args.no_amp, # Automatic Mixed Precision for 1.5x-2x speedup
        resume=True          # Added resume=True to resume from last checkpoint
    )

    elapsed_training = time.time() - start_time
    print(f"\n[SUCCESS] Model training completed in {format_time(elapsed_training)}!")

    # Perform Thesis Evaluation on Validation and Test Splits
    best_weights_path = run_dir / "weights" / "best.pt"
    if best_weights_path.exists():
        eval_model = YOLO(str(best_weights_path))
    else:
        eval_model = model

    print("\n" + "=" * 80)
    print(" THESIS EVALUATION: VALIDATION SET PERFORMANCE ")
    print("=" * 80)
    val_metrics = eval_model.val(data=str(data_path), split="val", imgsz=args.imgsz, plots=True)

    print("\n" + "=" * 80)
    print(" THESIS EVALUATION: TEST SET FINAL BENCHMARK ")
    print("=" * 80)
    test_metrics = eval_model.val(data=str(data_path), split="test", imgsz=args.imgsz, plots=True)

    # Print Summary Performance Metrics Table
    print("\n" + "=" * 80)
    print(" MASTER'S / PHD THESIS PERFORMANCE SUMMARY ")
    print("=" * 80)
    print(f"{'Split':<12} | {'Precision (P)':<15} | {'Recall (R)':<15} | {'mAP@50':<15} | {'mAP@50-95':<15}")
    print("-" * 80)

    val_p = val_metrics.results_dict.get("metrics/precision(B)", 0.0)
    val_r = val_metrics.results_dict.get("metrics/recall(B)", 0.0)
    val_map50 = val_metrics.results_dict.get("metrics/mAP50(B)", 0.0)
    val_map = val_metrics.results_dict.get("metrics/mAP50-95(B)", 0.0)

    test_p = test_metrics.results_dict.get("metrics/precision(B)", 0.0)
    test_r = test_metrics.results_dict.get("metrics/recall(B)", 0.0)
    test_map50 = test_metrics.results_dict.get("metrics/mAP50(B)", 0.0)
    test_map = test_metrics.results_dict.get("metrics/mAP50-95(B)", 0.0)

    print(f"{'Validation':<12} | {val_p:<15.4f} | {val_r:<15.4f} | {val_map50:<15.4f} | {val_map:<15.4f}")
    print(f"{'Test Set':<12} | {test_p:<15.4f} | {test_r:<15.4f} | {test_map50:<15.4f} | {test_map:<15.4f}")
    print("=" * 80)

    # Verify Generated Artifacts
    print("\n" + "=" * 80)
    print(" GENERATED THESIS FIGURES & ARTIFACTS VERIFICATION ")
    print("=" * 80)
    artifacts = verify_thesis_artifacts(run_dir)
    for artifact_name, status in artifacts:
        symbol = "[✓]" if status else "[×]"
        print(f" {symbol} {artifact_name}")
    print("=" * 80)

    # Print TensorBoard instructions
    print("\n" + "=" * 80)
    print(" 📊 TENSORBOARD MONITORING & VISUALIZATION INSTRUCTIONS ")
    print("=" * 80)
    print(f" TensorBoard has been launched automatically in the background.\n")
    print(f" You can access it via the 'Ports' tab in Google Colab (usually port 6006).\n")
    print(f" Direct access (if available): http://localhost:6006\n")
    print(" If you want to launch it manually later, run:")
    print(f"   tensorboard --logdir \"{log_dir}\" --host 0.0.0.0 --port 6006\n")
    print("=" * 80 + "\n")


def main() -> int:
    args = parse_args()
    run_yolo_training(args)
    return 0


if __name__ == "__main__":
    if _ultralytics_available:
        raise SystemExit(main())
    else:
        print("Training skipped due to missing 'ultralytics' library.")

In [ ]:
print('\n--- Verifying ultralytics installation ---')
!pip show ultralytics
print('------------------------------------------')

In [ ]:
print('\n--- Verifying ultralytics installation after re-install ---')
!pip show ultralytics
print('---------------------------------------------------------')